# CSC 2301 · Week 2 — Control flow on a well log

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W2_control_flow_well_log.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

Last week every line of a program ran once, from top to bottom. This week the program **decides**
(`if`) and **repeats** (`for`, `while`) on its own. The data are a gamma-ray and porosity log from
an exploration well, `KW-1` (synthetic, but realistic), sampled every 0.5 m from 1500 to 1650 m.

**What you will practise:** comparisons and `and` / `or` / `not` · `if` / `elif` / `else` ·
`for` with `range`, `zip` and `enumerate` · `while` · `break` and `continue` · counters and accumulators.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/well_log_kw1.csv', 'well_log_kw1.csv'), ('tests/w2/q1.py', 'tests/q1.py'), ('tests/w2/q2.py', 'tests/q2.py'), ('tests/w2/q3.py', 'tests/q3.py'), ('tests/w2/q4.py', 'tests/q4.py'), ('tests/w2/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. Reading the log

A **well log** records properties of the rocks along the borehole. We use two:

| column | meaning |
|---|---|
| `depth_m` | measured depth, in metres |
| `gr_api` | **gamma ray**, in API units: low in clean sand, high in shale (clay is radioactive) |
| `phi` | **porosity**, a fraction: 0.20 means 20 % of the rock is pore space |

We read the file into three lists of the same length: `depth[i]`, `gr[i]` and `phi[i]` describe the same sample.

In [ ]:
import csv

depth, gr, phi = [], [], []
with open("well_log_kw1.csv", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        depth.append(float(r["depth_m"]))
        gr.append(float(r["gr_api"]))
        phi.append(float(r["phi"]))

print(len(depth), "samples from", depth[0], "m to", depth[-1], "m")
print("First sample:", depth[0], gr[0], phi[0])

## 2. Comparisons and Boolean logic

A comparison gives `True` or `False`. Combine them with `and`, `or`, `not`.

In [ ]:
g, p = 45.0, 0.21
print(g < 60)                  # clean enough to be sand?
print(p >= 0.12)               # porous enough?
print(g < 60 and p >= 0.12)    # both
print(g > 100 or p < 0.05)     # at least one
print(not (g < 60))

## 3. Repeating with `for`

`range(n)` gives 0, 1, …, n−1. `zip` walks through several lists side by side, and `enumerate`
gives the position with each value.

In [ ]:
for i in range(3):
    print(i, depth[i], gr[i])

print("---")
for d, g in zip(depth[:3], gr[:3]):
    print(d, g)

print("---")
for i, g in enumerate(gr[:3]):
    print("sample", i, "GR =", g)

A loop often **accumulates** a result: a counter that goes up, a running total, a maximum so far.

In [ ]:
total = 0.0
highest = gr[0]
for g in gr:
    total += g             # same as total = total + g
    if g > highest:
        highest = g
print("Mean GR:", round(total / len(gr), 1), "API; highest:", highest)

## 4. `break` and `continue`

`break` leaves the loop at once; `continue` skips to the next value.
Example: the first sample deeper than 1600 m with a porosity above 25 %.

In [ ]:
found = None
for d, p in zip(depth, phi):
    if d <= 1600:
        continue           # not deep enough yet: skip
    if p > 0.25:
        found = d
        break              # stop at the first match
print("First very porous sample below 1600 m:", found)

## 5. Repeating with `while`

A `while` loop repeats **as long as** a condition is true. Use it when you do not know in advance
how many times to repeat. Example: a well produces 1200 barrels a day and declines by 15 % a year.
After how many years does it drop below 300 barrels a day, the economic limit?

In [ ]:
rate = 1200.0
years = 0
while rate >= 300:
    rate = rate * (1 - 0.15)
    years += 1
print("Below the economic limit after", years, "years; rate =", round(rate), "bbl/d")

---
## Exercises

Write your answer, run the cell, then run the `grader.check` cell below it.

### Q1. Counting sand samples

Geologists often call a sample **sand** when its gamma ray is below 60 API.
Store in `n_sand` the number of sand samples in the log.

In [ ]:
n_sand = ...
n_sand

In [ ]:
grader.check("q1")

### Q2. Shale volume

The **shale volume index** turns GR into a fraction between 0 (clean sand) and 1 (pure shale):

$$V_{sh} = \frac{GR - 30}{120 - 30}$$

Write a function `vshale(gr)` that returns this value, **clipped** to the range 0–1:
below 0 return 0.0, above 1 return 1.0.

In [ ]:
def vshale(gr):
    ...

[vshale(g) for g in (20, 30, 75, 120, 140)]

In [ ]:
grader.check("q2")

### Q3. First shale below 1540 m

Going down the well, find the first depth **deeper than 1540 m** where GR reaches 100 API or more.
Store it in `first_shale_depth`. Use a loop with `break` (and `continue` if you like).

In [ ]:
first_shale_depth = ...
first_shale_depth

In [ ]:
grader.check("q3")

### Q4. Net pay

A sample is **pay** (rock that can produce oil) when its shale volume is below 0.4 **and** its
porosity is at least 0.12. Each sample represents 0.5 m of rock.
Store in `net_pay_m` the total pay thickness in metres (a `float`). Use your `vshale`.

In [ ]:
net_pay_m = ...
net_pay_m

In [ ]:
grader.check("q4")

### Q5. How many sand beds?

Sand samples come in **beds**: runs of consecutive sand samples (GR < 60) separated by non-sand.
Store in `n_sand_beds` the number of separate sand beds.

*Hint:* keep a variable that remembers whether the previous sample was sand. A new bed starts when
the current sample is sand and the previous one was not.

In [ ]:
n_sand_beds = 0
...
n_sand_beds

In [ ]:
grader.check("q5")

### Going further (not checked)

* Print the top and base depth of each sand bed.
* With `while`: how many years until the well in section 5 drops below 300 bbl/d if the decline is 25 % a year?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()